# Lab 1: LLM Fine-tuning to Yoda style

This notebook fine-tunes [LiquidAI/LFM2-1.2B](https://huggingface.co/LiquidAI/LFM2-1.2B) to speak like Yoda, using LoRA. We then evaluate the result with an LLM-as-a-judge (Qwen3-Next via OpenRouter) tracked on Comet Opik.

**Differences from the original MIT template:**
1. Leprechaun stage removed — Yoda only.
2. Global random seed fixed and multi-seed averaging supported.
3. Custom `create_yoda_dataloaders` with 70/15/15 **train/val/test** split and `batch_size > 10`.
4. Full hyper-parameter sweep with results table & plots.
5. Three soft metrics: **Recall, BalancedAccuracy, StyleGain**.
6. Best LoRA weights are saved to `checkpoints/`.
7. Final `yoda_test_loglikelihood` cell is the unchanged competition metric.

All heavy logic lives under [`src/`](src/) — keep that directory next to the notebook.

## 0. Setup & dependencies

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os
import json
import sys
import pathlib

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Make local src/ importable
sys.path.insert(0, str(pathlib.Path('.').resolve()))

from src.utils import set_global_seed, TEMPLATE_WITH_ANSWER, format_question
from src.data import create_yoda_dataloaders, get_base_and_style_samples
from src.model import load_base_model_and_tokenizer, apply_lora, count_trainable_params, save_lora
from src.training import train, chat
from src.judge import make_yoda_judge, SYSTEM_PROMPT_BASIC, SYSTEM_PROMPT_DETAILED, YODA_EXAMPLE
from src.metrics import compute_style_metrics, format_metrics
from src.evaluation import evaluate_model, generate_samples, score_texts, yoda_loglikelihood

# Yoda test text from mit_dl_utils (the competition reference paragraph)
YODA_TEST_TEXT = (
    "Wisdom, sought by many, found by few, it is. Haste not, patience have. "
    "For in stillness, answers come. Much to learn, still you have. "
    "Fear leads to anger; anger, to hate. Down the dark path, guide you it will. "
    "Trust the Force, you must. Powerful ally it is. Life it creates, surrounds, binds. "
    "Adventure, excitement, a Jedi craves not these things. Discipline, balance, seek you should. "
    "Hmm, clearer now is the path, yes? Help you more, I can, if needed it is. "
    "Endless, the journey of learning is. Stay true to your path, and clarity you will find. "
    "Remember, the Force flows through all, but your heart determines how it shapes your destiny. "
    "Much more to teach, I have. Ready, are you? Mmm."
)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# Reproducibility -------------------------------------------------------------
SEED = 42
set_global_seed(SEED)

In [ ]:
# Opik tracking ---------------------------------------------------------------
# Load API keys from the environment (preferred) or set them inline.
OPIK_API_KEY = os.environ.get('OPIK_API_KEY', '')
OPIK_WORKSPACE = os.environ.get('OPIK_WORKSPACE', '')
OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')

if OPIK_API_KEY:
    import opik
    os.environ['OPIK_API_KEY'] = OPIK_API_KEY
    os.environ['OPIK_WORKSPACE'] = OPIK_WORKSPACE
    os.environ['OPIK_PROJECT_NAME'] = 'lab1-yoda'
    opik.configure()
    opik_client = opik.Opik()
    print('Opik configured.')
else:
    opik_client = None
    print('Opik NOT configured. Set OPIK_API_KEY to enable tracing.')

## 1. Data — train/val/test split (70/15/15, batch_size = 16)

In [ ]:
BATCH_SIZE = 16
train_loader, val_loader, test_loader = create_yoda_dataloaders(
    batch_size=BATCH_SIZE,
    train_ratio=0.70,
    val_ratio=0.15,
    seed=SEED,
)

print(f'train batches: {len(train_loader)} (batch_size={BATCH_SIZE})')
print(f'val   samples: {len(val_loader.dataset)}')
print(f'test  samples: {len(test_loader.dataset)}')

# Quick look
sample = train_loader.dataset[0]
print('--- Sample ---')
print('Instruction :', sample['instruction'][:140])
print('Base        :', sample['response'][:140])
print('Yoda style  :', sample['response_style'][:140])

## 2. Base model, tokenizer, baseline chat

In [ ]:
model, tokenizer = load_base_model_and_tokenizer()

In [ ]:
# Baseline inference BEFORE fine-tuning ---------------------------------------
baseline_questions = [
    'What is the capital of France?',
    'How do I train a neural network?',
    'Tell me a short story about courage.',
    'What is the meaning of patience?',
    'Explain photosynthesis to a child.',
]

baseline_answers = []
for q in baseline_questions:
    a = chat(model, tokenizer, q, max_new_tokens=60, temperature=0.3, only_answer=True)['answer']
    baseline_answers.append(a)
    print(f'Q: {q}\nA: {a}\n')

## 3. Apply LoRA

In [ ]:
LORA_R = 8
LORA_ALPHA = 16   # 2 * r
LORA_DROPOUT = 0.05

model = apply_lora(model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
stats = count_trainable_params(model)
print(f"trainable: {stats['trainable']:,}")
print(f"total    : {stats['total']:,}")
print(f"percent  : {stats['percent']:.2f}%")

## 4. Fine-tune

Default reference run: ~1 epoch over the training set, with a small validation check every 100 steps. Tweak `EPOCHS`, `LEARNING_RATE`, `CONTEXT_LENGTH` below for the sweep.

In [ ]:
EPOCHS = 1
LEARNING_RATE = 1e-4
CONTEXT_LENGTH = 512
MAX_STEPS = 400   # safety cap; set to None to use full epochs

set_global_seed(SEED)   # reset RNG just before training

def preview(step, avg_loss):
    out = chat(model, tokenizer, 'What is the capital of France?', max_new_tokens=24, only_answer=True)['answer']
    print(f"   preview[{step}]: {out!r}")

model, history = train(
    model,
    train_loader,
    val_loader,
    tokenizer,
    epochs=EPOCHS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    context_length=CONTEXT_LENGTH,
    preview_every=50,
    preview_fn=preview,
    log_val_every=100,
)

In [ ]:
# Loss curve
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history['steps'], history['train_loss'], label='train', alpha=0.6)
if history['val_loss']:
    vs, vl = zip(*history['val_loss'])
    ax.plot(vs, vl, label='val', marker='o', color='red')
ax.set_xlabel('step')
ax.set_ylabel('cross-entropy')
ax.set_title('Fine-tuning loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/loss_curve.png', dpi=120)
plt.show()

In [ ]:
# Save LoRA weights -----------------------------------------------------------
CKPT_DIR = 'checkpoints/yoda-lora-r8-lr1e-4'
save_lora(model, CKPT_DIR)
print('Saved LoRA adapter to', CKPT_DIR)

## 5. Inference AFTER fine-tuning

In [ ]:
tuned_answers = []
for q in baseline_questions:
    a = chat(model, tokenizer, q, max_new_tokens=60, temperature=0.3, only_answer=True)['answer']
    tuned_answers.append(a)
    print(f'Q: {q}\nA: {a}\n')

# Side-by-side comparison
compare = pd.DataFrame({
    'question': baseline_questions,
    'base_model': baseline_answers,
    'yoda_model': tuned_answers,
})
compare.to_csv('outputs/inference_before_after.csv', index=False)
compare

## 6. LLM-as-a-judge setup

Two system prompts are provided in `src/judge.py`:
* `SYSTEM_PROMPT_BASIC`    — the prompt from the MIT template.
* `SYSTEM_PROMPT_DETAILED` — a stricter prompt listing concrete Yoda-speech features.

Switch the `detailed` flag below to compare them.

In [ ]:
assert OPENROUTER_API_KEY, 'Set OPENROUTER_API_KEY (env var or here in the notebook).'
judge = make_yoda_judge(api_key=OPENROUTER_API_KEY, detailed=False)

# Vibe check
probes = [
    'Tennis is a fun sport. But you must concentrate.',                # base
    'Fun sport, tennis is. But work hard, you must.',                  # yoda-ish
    'Hard to see, the dark side is.',                                  # canonical Yoda
]
for t in probes:
    print(f'{judge.score(t).value:.2f}  <-  {t}')

## 7. Evaluation — generated vs base vs ground-truth Yoda

In [ ]:
N_EVAL = 20  # held-out prompts for evaluation
base_samples, style_samples = get_base_and_style_samples(test_loader, n_samples=N_EVAL)

results = evaluate_model(
    model,
    tokenizer,
    judge,
    test_loader,
    base_samples,
    style_samples,
    n_samples=N_EVAL,
    max_new_tokens=48,
    temperature=0.3,
)

print('=== Model (generated) vs Base ===')
print(format_metrics(results['model_metrics']))
print()
print('=== Ceiling (ground-truth Yoda) vs Base ===')
print(format_metrics(results['ceiling_metrics']))

In [ ]:
# Plots ----------------------------------------------------------------------
df = pd.DataFrame({
    'Score': [*results['base_scores'], *results['generated_scores'], *results['style_scores']],
    'Type':  (['Base'] * len(results['base_scores'])
            + ['Generated'] * len(results['generated_scores'])
            + ['Style'] * len(results['style_scores'])),
})
df.to_csv('outputs/scores.csv', index=False)

# Histogram
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x='Score', hue='Type', multiple='dodge', stat='probability', bins=6)
plt.xlabel('Judge score')
plt.ylabel('Probability')
plt.title('Distribution of judge scores')
plt.tight_layout()
plt.savefig('outputs/score_histogram.png', dpi=120)
plt.show()

# Bar chart with error bars
labels = ['Base', 'Generated', 'Style']
means = [np.mean(results['base_scores']), np.mean(results['generated_scores']), np.mean(results['style_scores'])]
stds  = [np.std(results['base_scores']),  np.std(results['generated_scores']),  np.std(results['style_scores'])]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, means, yerr=stds, capsize=6)
plt.ylim(0, 1)
plt.ylabel('Judge score')
plt.title('Mean judge score by text type')
for bar, value in zip(bars, means):
    plt.text(bar.get_x() + bar.get_width()/2, value + 0.02, f'{value:.2f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('outputs/score_bars.png', dpi=120)
plt.show()

## 8. Hyper-parameter sweep

Below is a programmatic sweep over `learning_rate`, `LoRA rank` and `context_length`. For each configuration we restart the model from the base checkpoint, re-apply LoRA, fine-tune, and evaluate. **Results are aggregated into a table and a bar plot.**

Toggle `RUN_SWEEP = True` to actually run it (takes time).

In [ ]:
RUN_SWEEP = False

SWEEP_CONFIGS = [
    {'lr': 1e-4, 'r': 8,  'ctx': 512},
    {'lr': 2e-4, 'r': 8,  'ctx': 512},
    {'lr': 1e-4, 'r': 16, 'ctx': 512},
    {'lr': 1e-4, 'r': 32, 'ctx': 512},
    {'lr': 1e-4, 'r': 8,  'ctx': 1024},
]

sweep_results = []

if RUN_SWEEP:
    for cfg in SWEEP_CONFIGS:
        set_global_seed(SEED)
        # fresh model for each run
        m, tok = load_base_model_and_tokenizer()
        m = apply_lora(m, r=cfg['r'], lora_alpha=2 * cfg['r'], lora_dropout=LORA_DROPOUT)
        m, hist = train(
            m, train_loader, val_loader, tok,
            epochs=1, max_steps=300,
            learning_rate=cfg['lr'],
            context_length=cfg['ctx'],
            preview_every=200,
            log_val_every=200,
        )
        bs, ss = get_base_and_style_samples(test_loader, n_samples=N_EVAL)
        res = evaluate_model(m, tok, judge, test_loader, bs, ss, n_samples=N_EVAL)
        ll = yoda_loglikelihood(m, tok, YODA_TEST_TEXT)
        save_lora(m, f"checkpoints/sweep_lr{cfg['lr']}_r{cfg['r']}_ctx{cfg['ctx']}")
        sweep_results.append({
            **cfg,
            'recall': res['model_metrics'].recall,
            'balanced_acc': res['model_metrics'].balanced_accuracy,
            'style_gain':   res['model_metrics'].style_gain,
            'yoda_ll':      ll,
        })
        del m
        torch.cuda.empty_cache()

    sweep_df = pd.DataFrame(sweep_results)
    sweep_df.to_csv('outputs/sweep_results.csv', index=False)
    print(sweep_df.to_string(index=False))
else:
    print('RUN_SWEEP is False — skipping. Set it to True to run the sweep.')

## 9. Multi-seed averaging (reproducibility check)

Run the best config across several seeds and report `mean ± std` of the soft metrics + the Yoda log-likelihood. Toggle `RUN_SEEDS = True` to actually run.

In [ ]:
RUN_SEEDS = False
SEEDS = [42, 7, 2025]

seed_records = []
if RUN_SEEDS:
    for s in SEEDS:
        set_global_seed(s)
        m, tok = load_base_model_and_tokenizer()
        m = apply_lora(m, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
        m, _ = train(m, train_loader, val_loader, tok,
                     epochs=1, max_steps=MAX_STEPS,
                     learning_rate=LEARNING_RATE,
                     context_length=CONTEXT_LENGTH,
                     preview_every=200,
                     log_val_every=200)
        bs, ss = get_base_and_style_samples(test_loader, n_samples=N_EVAL)
        res = evaluate_model(m, tok, judge, test_loader, bs, ss, n_samples=N_EVAL)
        ll = yoda_loglikelihood(m, tok, YODA_TEST_TEXT)
        seed_records.append({
            'seed': s,
            'recall': res['model_metrics'].recall,
            'balanced_acc': res['model_metrics'].balanced_accuracy,
            'style_gain': res['model_metrics'].style_gain,
            'yoda_ll': ll,
        })
        del m; torch.cuda.empty_cache()
    seed_df = pd.DataFrame(seed_records)
    seed_df.to_csv('outputs/seed_results.csv', index=False)
    print(seed_df.describe().loc[['mean', 'std']])
else:
    print('RUN_SEEDS is False — skipping.')

## 10. Yoda test log-likelihood  *(competition metric — DO NOT MODIFY)*

In [ ]:
# DO NOT CHANGE / MODIFY THIS CELL.
from torch.nn import functional as F

tokens = tokenizer(YODA_TEST_TEXT, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model(**tokens)
    logits = outputs.logits[:, :-1]
    targets = tokens.input_ids[:, 1:]
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

print(f'Yoda test loglikelihood: {loss.item():.2f}')

## 11. Conclusions (fill in after running)

- **Best config** (lr / r / ctx_len / epochs): ...
- **Yoda log-likelihood** (lower is better): ...
- **Soft Recall / Balanced Accuracy / Style Gain**: ...
- **Observations**: ...
- **Future improvements**: ...